# Deep Graph Infomax (Inductive PPI)

Unsupervised Representation on PPI: Maximizing mutual information between local node patches and global graph summary. This notebook implements the approach with `DeepGraphInfomax` inside a `SAGEEncoder` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DeepGraphInfomax` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models

title = "Deep Graph Infomax (Inductive Setup) with SAGEConv"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Encoder & Summary Function
class SAGEEncoder(keras.Model):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv = k3_layers.SAGEConv(in_channels, hidden_channels)

    def call(self, x, edge_index):
        return ops.relu(self.conv(x, edge_index))

def summary_fn(z, *args, **kwargs):
    return ops.sigmoid(ops.mean(z, axis=0))

def corruption_fn(x, edge_index):
    # Permute node features
    indices = keras.random.shuffle(ops.arange(ops.shape(x)[0]))
    return ops.take(x, indices, axis=0), edge_index

encoder = SAGEEncoder(in_channels=16, hidden_channels=32)
model = k3_models.DeepGraphInfomax(
    hidden_channels=32,
    encoder=encoder,
    summary=summary_fn,
    corruption=corruption_fn,
)

# 2. Sample Forward Pass
num_nodes = 50
dummy_x = keras.random.normal((num_nodes, 16))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

pos_z, neg_z, summary = model(dummy_x, dummy_edges)
print(f"DGI positive representations shape: {pos_z.shape}")
print(f"DGI summary representation shape: {summary.shape}")

print("\n✓ K3-Node DeepGraphInfomax (Inductive) execution completed successfully!")